<a href="https://colab.research.google.com/github/ksduong/GA-NP-Workforce-Map/blob/main/GA_NP_Workforce_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
from google.colab import files
up = files.upload()  # choose your CSV (e.g., 1103_np_address_with_county(in).csv)

Saving 1103_np_address_with_county(in).csv to 1103_np_address_with_county(in) (4).csv


In [27]:
import pandas as pd

INPUT_FILE = "1103_np_address_with_county(in).csv"  # <-- change if needed
np_df = pd.read_csv(INPUT_FILE)

print(np_df.shape)
np_df.head()

(14467, 19)


,NPI,Last_Name,First_Name,Sex,Street1,Street2,City,ZIP,Credentials,Certification_Date,Enumeration_Date,License_Number,NP_Type_Grouped,Taxonomy_Code,Taxonomy_Description,Medicare_Supplier_Type,Primary_Tax,Medicare_Specialty_Code,full_address
0,1003095977,NICHOLAS,TRACY,F,1968 PEACHTREE ROAD NW,PIEDMONT HOSPITAL,ATLANTA,30309,NP,NaN,10/29/2007,185022,Acute Care NP,363LA2100X,Physician Assistants & Advanced Practice Nursi...,Nurse Practitioner,Y,50,"1968 PEACHTREE ROAD NW PIEDMONT HOSPITAL, ATLA..."
1,1003104670,DODD,JOHN,M,2501 N PATTERSON SUITE,NaN,VALDOSTA,31602,NP,NaN,7/18/2011,RN117801,Acute Care NP,363LA2100X,Physician Assistants & Advanced Practice Nursi...,Nurse Practitioner,Y,50,"2501 N PATTERSON SUITE , VALDOSTA, GA"
2,1003128240,SARNO,JENNIFER,F,780 CANTON ROAD NE,SUITE 400,MARIETTA,30060,ACNP-BC,NaN,7/3/2010,177164,Acute Care NP,363LA2100X,Physician Assistants & Advanced Practice Nursi...,Nurse Practitioner,Y,50,"780 CANTON ROAD NE SUITE 400, MARIETTA, GA"
3,1003222191,SAUCEDO,ISRAEL,M,417 W 3RD AVENUE,NaN,ALBANY,31701,"APRN, FN",6/29/2022,7/9/2014,RN170526,Acute Care NP,363LA2100X,Physician Assistants & Advanced Practice Nursi...,Nurse Practitioner,N,50,"417 W 3RD AVENUE , ALBANY, GA"
4,1003280975,SHUMAKER-KIRK,CHARVETTE,F,600 CELEBRATE LIFE PARKWAY,NaN,NEWNAN,30265,NP-C,5/30/2023,11/16/2015,RN193221,Acute Care NP,363LA2100X,Physician Assistants & Advanced Practice Nursi...,Nurse Practitioner,Y,50,"600 CELEBRATE LIFE PARKWAY , NEWNAN, GA"


['NPI',
 'Last_Name',
 'First_Name',
 'Sex',
 'Street1',
 'Street2',
 'City',
 'ZIP',
 'Credentials',
 'Certification_Date',
 'Enumeration_Date',
 'License_Number',
 'NP_Type_Grouped',
 'Taxonomy_Code',
 'Taxonomy_Description',
 'Medicare_Supplier_Type',
 'Primary_Tax',
 'Medicare_Specialty_Code',
 'full_address']

In [39]:
# Start from your already-loaded DataFrame: np_df

# Rename to what our function expects
np_df = np_df.rename(columns={
    "Street1": "Street 1",
    "Street2": "Street 2",
    "ZIP": "Zip"
})

# Add a State column (since your CSV lacks one)
np_df["State"] = "GA"

# Clean Zip to 5 digits
np_df["Zip"] = np_df["Zip"].astype(str).str.extract(r"(\d{5})", expand=False).fillna("")

# Sanity check
list(np_df.columns)[:15], np_df[["Street 1","Street 2","City","State","Zip"]].head()

(['NPI',
  'Last_Name',
  'First_Name',
  'Sex',
  'Street 1',
  'Street 2',
  'City',
  'Zip',
  'Credentials',
  'Certification_Date',
  'Enumeration_Date',
  'License_Number',
  'NP_Type_Grouped',
  'Taxonomy_Code',
  'Taxonomy_Description'],
                      Street 1           Street 2      City State    Zip
 0      1968 PEACHTREE ROAD NW  PIEDMONT HOSPITAL   ATLANTA    GA  30309
 1      2501 N PATTERSON SUITE                NaN  VALDOSTA    GA  31602
 2          780 CANTON ROAD NE          SUITE 400  MARIETTA    GA  30060
 3            417 W 3RD AVENUE                NaN    ALBANY    GA  31701
 4  600 CELEBRATE LIFE PARKWAY                NaN    NEWNAN    GA  30265)

In [44]:
# PATCH: more robust Census response parsing + smaller batches

import io, requests, numpy as np, pandas as pd, csv, textwrap, time
from tqdm import trange

def add_county_from_address_via_census(np_df,
                                       street1="Street 1",
                                       street2="Street 2",
                                       city="City",
                                       state="State",
                                       zipcol="Zip",
                                       chunk_size=4000,       # smaller batches to be safe
                                       max_retries=3,
                                       pause_secs=4):
    def _parse_census_csv(text: str) -> pd.DataFrame:
        """Parse weird CSV safely; pad rows to max columns; error if HTML."""
        # If the service returned HTML (error page), show a snippet so we know why
        if text.lstrip().startswith("<"):
            preview = textwrap.shorten(text.replace("\n"," "), width=300)
            raise RuntimeError(f"Census returned HTML (likely error): {preview}")

        # Use Python's csv to avoid pandas tokenization errors
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if not rows:
            return pd.DataFrame()

        max_cols = max(len(r) for r in rows)
        # Pad short rows
        norm_rows = [r + [""]*(max_cols - len(r)) for r in rows]
        df = pd.DataFrame(norm_rows)

        # The batch geo format usually has at least 13 columns, but not guaranteed.
        # We'll rename known positions when present; otherwise fill with NaN.
        rename = {
            0:"id",
            2:"match",
            4:"matched_address",
            5:"lon",
            6:"lat",
            7:"tigerline_id",
            8:"side",
            9:"state_fips",
            10:"county_fips3",
            11:"tract",
            12:"block"
        }
        df = df.rename(columns={k:v for k,v in rename.items() if k < df.shape[1]})
        for col in rename.values():
            if col not in df.columns:
                df[col] = np.nan

        # Compose county_fips_5
        df["state_fips"] = df["state_fips"].astype(str).str.zfill(2)
        df["county_fips3"] = df["county_fips3"].astype(str).str.zfill(3)
        df["county_fips_5"] = np.where(
            df["state_fips"].str.len().eq(2) & df["county_fips3"].str.len().eq(3),
            df["state_fips"] + df["county_fips3"],
            np.nan
        )
        # Ensure ID exists
        if "id" not in df.columns:
            df["id"] = np.nan
        df["id"] = df["id"].astype(str)
        return df[["id","match","matched_address","lon","lat","state_fips","county_fips3","county_fips_5","tract","block"]]

    BATCH_URL = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"

    # --- Build batch rows (same as before)
    df = np_df.copy()
    df["__id"]   = df.index.astype(str)
    df["__zip5"] = df[zipcol].astype(str).str.extract(r"(\d{5})", expand=False).fillna("")
    df["__addr"] = (df[street1].fillna("").astype(str).str.strip() + " " +
                    df[street2].fillna("").astype(str).str.strip()).str.strip()

    batch_df = pd.DataFrame({
        "id": df["__id"],
        "address": df["__addr"].fillna(""),
        "city": df[city].fillna(""),
        "state": df[state].fillna(""),
        "zip": df["__zip5"]
    })
    batch_df = batch_df[ batch_df["address"].str.len().ge(3) & batch_df["state"].str.len().ge(2) ].copy()

    results = []
    starts = list(range(0, len(batch_df), chunk_size))
    for i in trange(len(starts), desc="Geocoding batches"):
        start = starts[i]
        chunk = batch_df.iloc[start:start+chunk_size]
        buf = io.StringIO()
        chunk.to_csv(buf, index=False, header=False)  # NO HEADER

        # retry loop
        for attempt in range(1, max_retries+1):
            try:
                r = requests.post(
                    BATCH_URL,
                    files={"addressFile": ("addresses.csv", buf.getvalue().encode("utf-8"), "text/csv")},
                    data={"benchmark":"Public_AR_Current","vintage":"Current_Current"},
                    timeout=180
                )
                r.raise_for_status()
                results.append(_parse_census_csv(r.text))
                break
            except Exception as e:
                if attempt == max_retries:
                    raise
                time.sleep(pause_secs)

    geo = pd.concat(results, ignore_index=True) if results else pd.DataFrame(columns=["id","county_fips_5"])
    out = df.merge(geo, left_on="__id", right_on="id", how="left")
    out["is_ga_county"] = out["county_fips_5"].str.startswith("13", na=False)
    return out.drop(columns=["__id","__zip5","__addr","id"])

In [45]:
np_geo = add_county_from_address_via_census(np_df)
print(np_geo.shape)
np_geo.head()

Geocoding batches: 100%|██████████| 4/4 [03:31<00:00, 52.77s/it]

(14467, 30)


,NPI,Last_Name,First_Name,Sex,Street 1,Street 2,City,Zip,Credentials,Certification_Date,...,match,matched_address,lon,lat,state_fips,county_fips3,county_fips_5,tract,block,is_ga_county
0,1003095977,NICHOLAS,TRACY,F,1968 PEACHTREE ROAD NW,PIEDMONT HOSPITAL,ATLANTA,30309,NP,NaN,...,Match,"1968 PEACHTREE ST NW, ATLANTA, GA, 30309","-84.393664726494,33.808406957224",618355852,121,009104,NaN,2003,NaN,False
1,1003104670,DODD,JOHN,M,2501 N PATTERSON SUITE,NaN,VALDOSTA,31602,NP,NaN,...,Match,"2501 N PATTERSON ST, VALDOSTA, GA, 31602","-83.28881183844,30.861670540771",87740715,185,010401,NaN,2004,NaN,False
2,1003128240,SARNO,JENNIFER,F,780 CANTON ROAD NE,SUITE 400,MARIETTA,30060,ACNP-BC,NaN,...,Match,"780 CANTON RD NE, MARIETTA, GA, 30060","-84.547800829213,33.971033586059",650708908,067,030700,NaN,1003,NaN,False
3,1003222191,SAUCEDO,ISRAEL,M,417 W 3RD AVENUE,NaN,ALBANY,31701,"APRN, FN",6/29/2022,...,Match,"417 3RD AVE W, ALBANY, GA, 31701","-84.156564208956,31.59028532575",84324266,095,011300,NaN,1055,NaN,False
4,1003280975,SHUMAKER-KIRK,CHARVETTE,F,600 CELEBRATE LIFE PARKWAY,NaN,NEWNAN,30265,NP-C,5/30/2023,...,No_Match,,,,00,000,00000,,NaN,False


from matplotlib import pyplot as plt
_df_0['NPI'].plot(kind='hist', bins=20, title='NPI')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_1.groupby('Last_Name').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_15['First_Name'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_15, x='NPI', y='First_Name', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_16['Sex'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_16, x='NPI', y='Sex', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_17['Street 1'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_17, x='NPI', y='Street 1', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [46]:
np_geo.head()
np_geo.columns

Index(['NPI', 'Last_Name', 'First_Name', 'Sex', 'Street 1', 'Street 2', 'City',
       'Zip', 'Credentials', 'Certification_Date', 'Enumeration_Date',
       'License_Number', 'NP_Type_Grouped', 'Taxonomy_Code',
       'Taxonomy_Description', 'Medicare_Supplier_Type', 'Primary_Tax',
       'Medicare_Specialty_Code', 'full_address', 'State', 'match',
       'matched_address', 'lon', 'lat', 'state_fips', 'county_fips3',
       'county_fips_5', 'tract', 'block', 'is_ga_county'],
      dtype='object')

In [47]:
found = np_geo["county_fips_5"].notna().sum()
print(f"County FIPS found for {found:,} of {len(np_geo):,} rows ({found/len(np_geo):.1%}).")

np_geo.to_csv("np_addresses_with_county_fips.csv", index=False)

from google.colab import files
files.download("np_addresses_with_county_fips.csv")

County FIPS found for 2,032 of 14,467 rows (14.0%).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
import pandas as pd
INPUT_FILE = "1103_np_address_with_county(in).csv"  # change if needed
np_raw = pd.read_csv(INPUT_FILE)
np_raw.shape, np_raw.head(2)

((14467, 19),
           NPI Last_Name First_Name Sex                 Street1  \
 0  1003095977  NICHOLAS      TRACY   F  1968 PEACHTREE ROAD NW   
 1  1003104670      DODD       JOHN   M  2501 N PATTERSON SUITE   
 
              Street2      City    ZIP Credentials Certification_Date  \
 0  PIEDMONT HOSPITAL   ATLANTA  30309          NP                NaN   
 1                NaN  VALDOSTA  31602          NP                NaN   
 
   Enumeration_Date License_Number NP_Type_Grouped Taxonomy_Code  \
 0       10/29/2007         185022   Acute Care NP    363LA2100X   
 1        7/18/2011       RN117801   Acute Care NP    363LA2100X   
 
                                 Taxonomy_Description Medicare_Supplier_Type  \
 0  Physician Assistants & Advanced Practice Nursi...     Nurse Practitioner   
 1  Physician Assistants & Advanced Practice Nursi...     Nurse Practitioner   
 
   Primary_Tax  Medicare_Specialty_Code  \
 0           Y                       50   
 1           Y              

In [49]:
import numpy as np, io, requests, csv, textwrap, time
from tqdm import trange

df = np_raw.copy()  # keep np_raw untouched

# Non-destructive helpers (don’t overwrite your originals)
df["__rowid"] = df.index.astype(str)
# Your file had: Street1, Street2, City, ZIP (no State). We'll *add* State = 'GA' (adjust if needed)
street1 = df.get("Street1", df.get("Street 1", ""))
street2 = df.get("Street2", df.get("Street 2", ""))
city    = df.get("City", "")
state   = df.get("State", "GA")  # add GA if missing
zip_raw = df.get("ZIP", df.get("Zip",""))

# Cleaned copies (don’t touch originals)
df["__addr"] = (street1.fillna("").astype(str).str.strip()+" "+street2.fillna("").astype(str).str.strip()).str.strip()
df["__city"] = city.fillna("").astype(str)
df["__state"] = (state if isinstance(state, pd.Series) else pd.Series(state, index=df.index)).astype(str)
df["__zip5"] = pd.Series(zip_raw).astype(str).str.extract(r"(\d{5})", expand=False).fillna("")

In [63]:
def _parse_census_csv(text: str) -> pd.DataFrame:
    # Detect HTML error pages
    if text.lstrip().startswith("<"):
        preview = textwrap.shorten(text.replace("\n"," "), width=300)
        raise RuntimeError(f"Census returned HTML (likely rate/format issue): {preview}")
    rows = list(csv.reader(io.StringIO(text)))
    if not rows: return pd.DataFrame(columns=["id"])
    maxc = max(len(r) for r in rows)
    rows = [r + [""]*(maxc-len(r)) for r in rows]
    out = pd.DataFrame(rows)
    # Map known columns when present
    rename = {0:"id", 2:"match", 4:"matched_address", 5:"lon", 6:"lat", 7:"tigerline_id", 8:"side", 9:"state_fips", 10:"county_fips3", 11:"tract", 12:"block"}
    out = out.rename(columns={k:v for k,v in rename.items() if k < out.shape[1]})
    for v in rename.values():
        if v not in out.columns: out[v] = np.nan
    out["state_fips"] = out["state_fips"].astype(str).str.zfill(2)
    out["county_fips3"] = out["county_fips3"].astype(str).str.zfill(3)
    out["county_fips_5"] = np.where(out["state_fips"].str.len().eq(2) & out["county_fips3"].str.len().eq(3),
                                    out["state_fips"] + out["county_fips3"], np.nan)
    return out[["id","match","matched_address","lon","lat","state_fips","county_fips3","county_fips_5","tract","block"]]

def census_geocode_batches(payload_df, chunk_size=4000, retry=3, pause=4):
    BATCH_URL = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"
    results = []
    starts = range(0, len(payload_df), chunk_size)
    for i, start in enumerate(starts):
        chunk = payload_df.iloc[start:start+chunk_size]
        buf = io.StringIO()
        # CSV must be: id,address,city,state,zip with NO header
        chunk_to_send = pd.DataFrame({
            "id": chunk["__rowid"],
            "address": chunk["__addr"],
            "city": chunk["__city"],
            "state": chunk["__state"],
            "zip": chunk["__zip5"]
        })
        chunk_to_send.to_csv(buf, index=False, header=False)

        for attempt in range(1, retry+1):
            try:
                r = requests.post(
                    BATCH_URL,
                    files={"addressFile": ("addresses.csv", buf.getvalue().encode("utf-8"), "text/csv")},
                    data={"benchmark":"Public_AR_Current","vintage":"Current_Current"},
                    timeout=180
                )
                r.raise_for_status()
                results.append(_parse_census_csv(r.text))
                break
            except Exception as e:
                if attempt == retry:
                    raise
                time.sleep(pause)
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame(columns=["id"])